# Logistic Regression & Bayes Classification

We turn the disaster data into a **classification** task — predict whether an event is *Volcanic activity* from where it occurred — and meet logistic regression, LDA, QDA and Naïve Bayes.

## How this notebook works

<table>
<tr><td style="font-size:1.4em;">🧮</td><td><b style="color:#E37222;">Level 1 · By hand</b></td><td>Do the math yourself on small <b>mock data</b> — plain arithmetic, no libraries.</td></tr>
<tr><td style="font-size:1.4em;">🐍</td><td><b style="color:#0065BD;">Level 2 · With NumPy</b></td><td>Implement the formula with basic <b>NumPy</b> on the real data.</td></tr>
<tr><td style="font-size:1.4em;">📚</td><td><b style="color:#5E6A00;">Level 3 · With libraries</b></td><td>Use <b>pandas / scikit-learn</b> — the practical way.</td></tr>
</table>

Not every topic offers all three levels — the foundational ones do, later applied ones jump straight to the practical level. When stuck, expand a hint: <b>💡 Hint</b> names the functions (with docs); <b>🔑 Stuck?</b> shows them on mock data.

## Setup &mdash; packages and data

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use(os.path.join(os.getcwd(), "..", "..", "configs", "visualisations.mplstyle"))
df = pd.read_csv(os.path.join("..", "..", "data", "1900_2021_DISASTERS.csv"))

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Disaster Type'] = df['Disaster Type'].astype('category')

focus = 'Volcanic activity'
sub = df.dropna(subset=['Latitude', 'Longitude']).copy()
sub['Distance to Equator'] = sub['Latitude'].abs()
sub['abs Longitude'] = sub['Longitude'].abs()
sub[f'P({focus})'] = (sub['Disaster Type'] == focus).astype(int)
pos = sub[sub[f'P({focus})'] == 1]
neg = sub[sub[f'P({focus})'] == 0].sample(n=len(pos), random_state=0)
df_plot = pd.concat([pos, neg]).reset_index(drop=True)
df_plot = df_plot[df_plot['abs Longitude'] < 180]
x_col = 'Distance to Equator'
y_col = f'P({focus})'
df_plot.head()

,Year,Seq,Glide,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,Disaster Subsubtype,Event Name,Country,...,Insured Damages ('000 US$),Total Damages ('000 US$),CPI,Adm Level,Admin1 Code,Admin2 Code,Geo Locations,Distance to Equator,abs Longitude,P(Volcanic activity)
0,1976,77,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Cotopaxi,Ecuador,...,NaN,NaN,21.988357,NaN,NaN,NaN,NaN,0.677,78.436,1
1,1906,12,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Mt. Vesuve,Italy,...,NaN,NaN,3.479379,NaN,NaN,NaN,NaN,40.821,14.426,1
2,1944,44,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Mt. Vesuve,Italy,...,NaN,NaN,6.817006,NaN,NaN,NaN,NaN,40.821,14.426,1
3,1957,11,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Manam,Papua New Guinea,...,NaN,NaN,10.854113,NaN,NaN,NaN,NaN,4.080,145.037,1
4,1960,11,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Manam,Papua New Guinea,...,NaN,NaN,11.427251,NaN,NaN,NaN,NaN,4.080,145.037,1


# Logistic regression

### Why not linear regression?

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Fit a degree-1 linear regression of the 0/1 target on <code>Distance to Equator</code> and predict over a wide range. Show that the 'probabilities' fall outside [0, 1].</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit with [`np.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)`(x, y, 1)`, evaluate with [`np.poly1d`](https://numpy.org/doc/stable/reference/generated/numpy.poly1d.html) over [`np.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html)`(-50, 200, 200)`.

</details>

In [ ]:
x = df_plot[x_col].values
y = df_plot[y_col].values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f"linear predictions range: [{y_lin.min():.2f}, {y_lin.max():.2f}]  (a probability should stay in [0,1])")
plt.scatter(x, y, s=30, alpha=0.6, edgecolors='w', linewidth=0.5)
plt.plot(x_grid, y_lin, color='red', lw=2, label='linear fit')
plt.axhline(0, color='grey', ls='--', lw=0.8); plt.axhline(1, color='grey', ls='--', lw=0.8)
plt.xlabel(x_col); plt.ylabel(y_col); plt.legend(); plt.title('Linear regression on a 0/1 target'); plt.show()

### The sigmoid link
The logistic function $\sigma(z) = e^z/(1+e^z)$ squashes any real number into $[0,1]$.

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">Given e² ≈ 7.389, compute σ(0) and σ(2) by hand. (σ(0) = e⁰/(1+e⁰); σ(2) = e²/(1+e²).)</span></div>

In [ ]:
# sigmoid(z) = e^z / (1 + e^z),  with e^0 = 1 and e^2 ≈ 7.389

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'sigmoid(0) = {sig_0:.3f}, sigmoid(2) = {sig_2:.3f}')

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement the logistic function with NumPy and confirm it stays in [0, 1] and matches <code>scipy.special.expit</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`z = b0 + b1*X`, then `np.exp(z) / (1 + np.exp(z))` using [`np.exp`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html); compare with [`scipy.special.expit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.expit.html).

</details>

In [ ]:
from scipy.special import expit
beta0, beta1 = 1.0, -0.08

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

x_grid = np.linspace(-50, 200, 200)
p = logistic(x_grid, beta0, beta1)
print('stays in [0, 1]?  ', bool(p.min() >= 0 and p.max() <= 1))
print('matches scipy.expit?', np.allclose(p, expit(beta0 + beta1 * x_grid)))
plt.scatter(x, y, s=30, alpha=0.6, edgecolors='w', linewidth=0.5)
plt.plot(x_grid, p, color='purple', lw=2, label='logistic $P(X)$')
plt.xlabel(x_col); plt.ylabel(y_col); plt.legend(); plt.title('The logistic link keeps P(X) in [0, 1]'); plt.show()

### Fit logistic regression & interpret the coefficient

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Fit a logistic regression with <code>statsmodels.Logit</code>, extract β₁, and convert it to an odds ratio e^β₁.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Add an intercept with [`sm.add_constant`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html), fit with [`sm.Logit`](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html)`(y, X).fit(disp=0)`; odds ratio = [`np.exp`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html)`(b1)`.

</details>

In [ ]:
import statsmodels.api as sm
X1 = sm.add_constant(df_plot[[x_col]])
y1 = df_plot[y_col]

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'b1 = {b1:.4f}')
print(f'odds ratio exp(b1) = {odds_ratio:.4f}  (multiplicative change in odds per +1 degree)')

### Maximum likelihood

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Logistic regression maximizes the log-likelihood Σ [y·log p + (1−y)·log(1−p)]. Implement it with NumPy and verify the fitted β scores higher than an all-zero guess.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`p = expit(Xmat @ beta)`; then [`np.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)`(y*`[`np.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html)`(p) + (1-y)*np.log(1-p))`.

</details>

In [ ]:
beta_fit = logit_res.params.values
beta_zero = np.array([0.0, 0.0])
Xmat = X1.values.astype(float)
yv = y1.values.astype(float)

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'log-likelihood at fitted beta = {log_likelihood(beta_fit, Xmat, yv):.2f}')
print(f'log-likelihood at zero beta   = {log_likelihood(beta_zero, Xmat, yv):.2f}')
print('the MLE should give the higher (less negative) value')

### Multiple logistic regression

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Add a second predictor (<code>abs Longitude</code>) and fit a multiple logistic regression. Report the coefficients and the training accuracy at a 0.5 threshold.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Same [`sm.Logit`](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html) call with two predictors; threshold `res.predict(X) > 0.5` and compare to the labels.

</details>

In [ ]:
features = ['Distance to Equator', 'abs Longitude']
Xm = sm.add_constant(df_plot[features])
ym = df_plot[y_col]

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print('coefficients:')
print(res_multi.params)
print(f'training accuracy = {accuracy:.3f}')

# Discriminant analysis

### Linear Discriminant Analysis (LDA) from Bayes' rule

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Build a 1-D LDA by hand: estimate the priors π_k = n_k/n, class means μ_k, and one shared variance σ². Classify each point by the larger posterior π_k·N(x | μ_k, σ²), and compare to scikit-learn's LDA.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`np.unique`/masks give the per-class subsets; the Gaussian density is [`scipy.stats.norm.pdf`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html); compare to [`LinearDiscriminantAnalysis`](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html).

</details>

In [ ]:
from scipy.stats import norm
xx = df_plot[x_col].values
yy = df_plot[y_col].values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
sk = LinearDiscriminantAnalysis().fit(xx.reshape(-1, 1), yy)
print(f'your LDA accuracy    = {(pred_lda == yy).mean():.3f}')
print(f'sklearn LDA accuracy = {sk.score(xx.reshape(-1, 1), yy):.3f}')

### Quadratic Discriminant Analysis (QDA)

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Fit both LDA and QDA on two features and plot their decision boundaries side by side — LDA's is linear, QDA's is curved.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

[`LinearDiscriminantAnalysis`](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html) and [`QuadraticDiscriminantAnalysis`](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.QuadraticDiscriminantAnalysis.html); predict over a [`np.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html) and shade with `contourf`.

</details>

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
features = ['Distance to Equator', 'abs Longitude']
Xq = df_plot[features].values
yq = df_plot[y_col].values

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'LDA accuracy = {acc_lda:.3f}')
print(f'QDA accuracy = {acc_qda:.3f}')
xxg, yyg = np.meshgrid(np.linspace(0, 90, 200), np.linspace(0, 180, 200))
grid = np.c_[xxg.ravel(), yyg.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, clf, t in [(axes[0], lda, 'LDA (linear boundary)'), (axes[1], qda, 'QDA (curved boundary)')]:
    zz = clf.predict(grid).reshape(xxg.shape)
    ax.contourf(xxg, yyg, zz, alpha=0.2, cmap='coolwarm')
    ax.scatter(Xq[:, 0], Xq[:, 1], c=yq, cmap='coolwarm', s=20, edgecolors='w', linewidth=0.3)
    ax.set_xlabel('Distance to Equator'); ax.set_title(t)
axes[0].set_ylabel('abs Longitude'); plt.tight_layout(); plt.show()

# Naïve Bayes

### Gaussian Naïve Bayes

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Gaussian Naïve Bayes is QDA with the off-diagonal covariance forced to zero (features assumed independent). Fit <code>GaussianNB</code> on the same two features and compare its accuracy to QDA.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Fit [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) and call `.score` — compare to the QDA accuracy from above.

</details>

In [ ]:
from sklearn.naive_bayes import GaussianNB

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

print(f'Naive Bayes accuracy = {acc_nb:.3f}')
print(f'(QDA accuracy was {acc_qda:.3f})')
print('Naive Bayes assumes features are independent within a class -> diagonal covariance.')